# 结构化输出 structured_output
models 模型可以请求LLM提供格式化后的响应，通过给一个 schema。
这对于确保输出是被简单切分过得，可以直接被使用在后续的过程中。
LangChain提供了多个  schema 类型和方法来强制结构化的输出。

先创建一个Model

In [1]:
import os

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

## Pydantic
Pydantic Models 提供了最丰富的特性通过字段校验，描述和嵌套结构。

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")


In [ ]:
from langchain_core.messages import HumanMessage
model_with_structure = model.with_structured_output(Movie)

# ✅ 强制模型返回完整JSON（关键！） 为了适配千问，其中之一的解决办法：
# 提示词必须强制要求返回所有字段
# 必须包含 Return in valid JSON format.
prompt = """
Extract the information about the movie 'Inception'.
You MUST return a valid JSON object with ALL fields:
- title: string
- year: integer (only number)
- director: string
- rating: FLOAT (number between 0-10, like 8.2 or 9.0)
DO NOT return content rating like PG-13.
Return ONLY JSON.
"""

response = model_with_structure.invoke([
    HumanMessage(content=prompt)
])
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8


#### 千问模型 针对性处理
method="json_mode" 必须显式指定：千问不支持原生 json_schema，必须用 json_mode 适配百炼的 JSON Mode
提示词必须包含 JSON：否则直接触发 400 报错，这是百炼的强制校验规则

In [6]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

from pydantic import BaseModel, Field


class Movie(BaseModel):
    """A movie with details"""

    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")


model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),

    model_kwargs={"response_format": {"type": "json_object"}}, # 必需
)

model_with_structure = model.with_structured_output(Movie, method="json_mode")
prompt = """
Provide details about the movie Inception.
Return a valid JSON object with ALL required fields: title, year, director, rating.
rating must be a float number between 0-10, DO NOT return content rating like PG-13.
"""

# 5. 调用（最新版标准写法）
response = model_with_structure.invoke([HumanMessage(content=prompt)])
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8


### include_raw
模型返回结果后，我们一般只拿到 结构化数据（比如 Movie 对象），
但原始的完整消息（AIMessage）里还藏着很多有用信息（用了多少 token、耗时、模型版本等）。
如果开启 include_raw=True，就能同时拿到两样东西：
你要的结构化数据
原始消息（含元数据）

In [7]:
model_with_structure = model.with_structured_output(Movie, method="json_mode", include_raw=True)
prompt = """
Provide details about the movie Inception.
Return a valid JSON object with ALL required fields: title, year, director, rating.
rating must be a float number between 0-10, DO NOT return content rating like PG-13.
"""

# 5. 调用（最新版标准写法）
response = model_with_structure.invoke([HumanMessage(content=prompt)])
print(response)

{'raw': AIMessage(content='{\n  "title": "Inception",\n  "year": 2010,\n  "director": "Christopher Nolan",\n  "rating": 8.8\n}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13296, 'prompt_tokens': 64, 'total_tokens': 13360, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 13251, 'rejected_prediction_tokens': None, 'text_tokens': 13296}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 64}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus', 'system_fingerprint': None, 'id': 'chatcmpl-610441a0-6442-948e-b896-9724e930bdb3', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d579f-bf8f-7f60-8536-ede8cdbcea35-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 13296, 'total_tokens': 13360, 'input_token_details': {}, 'output_token_details': {'reasoning': 13251}}), 'parsed': M

### 嵌套结构 Nested structures


In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, method="json_mode")
prompt = """
Provide the official details for the movie 'Inception' in valid JSON format.
You MUST return ALL of these fields exactly:

- title: string
- year: integer (release year)
- cast: list of objects, each with "name" (actor name) and "role" (character name)
- genres: list of strings (movie genres)
- budget: number or null (in millions USD)

Return ONLY clean JSON, no extra text.
"""

# 5. 调用（最新版标准写法）
response = model_with_structure.invoke([HumanMessage(content=prompt)])
print(response)

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Miles')] genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'] budget=160.0
